In [3]:
import sqlite3
import pandas as pd
import numpy as np
from tqdm import tqdm
import json

In [4]:
SQLITE_PATH = "dev_ai_articles_full.sqlite"


dev_articles = "articles"
DEV_COLUMN_CONFIG = {
    "id": "id",           # primary key column
    "title": "title",     # article title column
    "body": "body_text",       # article body/content column (set to None if not available)
}

hashnode_articles = "hashnode_articles"
HASHNODE_COLUMN_CONFIG = {
    "id": "id",           # primary key column
	"title": "title",     # article title column
	"body": "body",       # article body/content column (set to None if not available)
}

# Output table where results will be written
OUTPUT_TABLE = "article_classifications"

# How much body text to use (tokens are expensive; first 500 chars is usually enough)
BODY_PREVIEW_CHARS = 500

# Batch size for zero-shot classification (lower if you run out of RAM)
CLASSIFICATION_BATCH_SIZE = 32

In [5]:
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()

In [6]:
id_col = DEV_COLUMN_CONFIG["id"]
title_col = DEV_COLUMN_CONFIG["title"]
body_col = DEV_COLUMN_CONFIG["body"]

query = f"SELECT {id_col}, {title_col}, {body_col} FROM {dev_articles}"

df = pd.read_sql_query(query, conn)


hashnode_df = pd.read_sql_query(f"SELECT {id_col}, {title_col}, {body_col} FROM {hashnode_articles}", conn)




In [9]:
df

,id,title,body_text
0,142249,The Distressed Code Review,title: The Distressed Code Review published: t...
1,198370,Avoid These Terrible Web Notifications Mistake...,This article is a brief introduction to some b...
2,385335,DNS Explained. Resolution,This is an article in the DNS Explained. serie...
3,423055,You don't need a library for state machines,title: You don't need a library for state mach...
4,441453,Data Engineering Series #3: Apache Airflow - t...,Why such attention towards Airflow ? Interest ...
...,...,...,...
3019,3363277,Hardware Hacking 101 Village - Post-Mortem,title: Hardware Hacking 101 Village Post Morte...
3020,3363288,FE/BE - Unite Them!,tl;dr; Teams should agree upon and understand ...
3021,3363939,Announcing the Colab MCP Server: Connect Any A...,When you’re prototyping locally with AI agents...
3022,3364128,I Built a Claude Code Agent That Doesn't Need ...,title: I Built a Claude Code Agent That Doesn'...


In [10]:
hashnode_df

,id,title,body_text
0,69c4b8a8efeaf33e6b3675be,MonALISA : A Distributed Monitoring Service Ar...,Adaptive Monitoring for Large Scale Grids: a S...
1,69c4bd952d879655ece508e5,The Sweet Spot of AI Orchestration: From AWS L...,The Sweet Spot of AI Orchestration: From AWS L...
2,69c4aaa5bbdb1bc33f100e25,"45 Claude Code Hooks for Code Quality, Securit...",45 Claude Code Hooks I Use to Automate Code Qu...
3,69c4a9faaed4ec64674bd0db,Claude Code Channels: Set Up a 24/7 AI Agent v...,Claude Code Channels Just Dropped — Here's How...
4,69c4a88eea32df99834d3704,I Built My Own OpenClaw Alternative With Claud...,I Built My Own OpenClaw Alternative With Claud...
...,...,...,...
22643,69a85632e55311e40f0eeb63,Building an AI-Powered Hospital Infection Inte...,A Case Study from CODE:AUTOMATA Ver 2.1 Hackat...
22644,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...
22645,69a8589de55311e40f1023c3,How AI Is Changing Software Architecture Planning,Most AI conversations in software focus on cod...
22646,69a6cd5c75d7a0f10015ca0d,How to Run and Customize LLMs Locally with Ollama,In the long history of technological innovatio...


In [11]:
df_all = pd.concat([df, hashnode_df], ignore_index=True)

In [12]:
df_all

,id,title,body_text
0,142249,The Distressed Code Review,title: The Distressed Code Review published: t...
1,198370,Avoid These Terrible Web Notifications Mistake...,This article is a brief introduction to some b...
2,385335,DNS Explained. Resolution,This is an article in the DNS Explained. serie...
3,423055,You don't need a library for state machines,title: You don't need a library for state mach...
4,441453,Data Engineering Series #3: Apache Airflow - t...,Why such attention towards Airflow ? Interest ...
...,...,...,...
25667,69a85632e55311e40f0eeb63,Building an AI-Powered Hospital Infection Inte...,A Case Study from CODE:AUTOMATA Ver 2.1 Hackat...
25668,69a8580ae55311e40f0fd641,You don't know your AI stack.,AI tooling has no supply chain. No audit trail...
25669,69a8589de55311e40f1023c3,How AI Is Changing Software Architecture Planning,Most AI conversations in software focus on cod...
25670,69a6cd5c75d7a0f10015ca0d,How to Run and Customize LLMs Locally with Ollama,In the long history of technological innovatio...


Now that dfs are combined, we can start clustering

In [13]:
df_all["combined_text"] = (
	df_all[title_col].fillna("") + " " +
	df_all[body_col].fillna("").str[:BODY_PREVIEW_CHARS]
).str.strip()

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, fast, good quality

texts = df_all["combined_text"].tolist()
embeddings = model.encode(
	texts,
	batch_size=64,
	show_progress_bar=True,
	convert_to_numpy=True,
)
print(f"✅ Embeddings shape: {embeddings.shape}")


embeddings

Batches: 100%|██████████| 402/402 [07:37<00:00,  1.14s/it]

✅ Embeddings shape: (25672, 384)


array([[-0.08685587,  0.01264643, -0.02058709, ..., -0.05263602,
         0.04710007,  0.07132634],
       [ 0.01738494, -0.05601176,  0.00476142, ...,  0.00340776,
        -0.05508111,  0.06832714],
       [ 0.00640805,  0.00613374,  0.02814993, ...,  0.04672879,
        -0.01237991, -0.01386962],
       ...,
       [-0.00100858,  0.04057033,  0.0403581 , ..., -0.0136122 ,
         0.01158926, -0.02284624],
       [ 0.01319809, -0.0618362 ,  0.02602076, ..., -0.00230252,
        -0.03536336, -0.01943028],
       [-0.04197986, -0.03975216, -0.01501757, ..., -0.00815797,
         0.11252944, -0.04953219]], shape=(25672, 384), dtype=float32)

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

n_clusters = 10


# Reduce dims first for speed
pca = PCA(n_components=50, random_state=42)
reduced = pca.fit_transform(embeddings)

km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=3)
labels = km.fit_predict(reduced)

unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
	print(f"   Cluster {u:2d}: {c:,} articles")


labels

   Reducing dimensions with PCA...
   Running k-means...
   Cluster  0: 2,138 articles
   Cluster  1: 1,206 articles
   Cluster  2: 2,727 articles
   Cluster  3: 2,461 articles
   Cluster  4: 2,379 articles
   Cluster  5: 2,448 articles
   Cluster  6: 3,311 articles
   Cluster  7: 2,044 articles
   Cluster  8: 3,290 articles
   Cluster  9: 3,668 articles


array([2, 2, 6, ..., 5, 7, 5], shape=(25672,), dtype=int32)

In [22]:
n_samples = 5
df_all_copy = df_all.copy()
df_all_copy["cluster"] = labels
for cluster_id in sorted(df_all_copy["cluster"].unique()):
	samples = df_all_copy[df_all_copy["cluster"] == cluster_id]["combined_text"].head(n_samples).tolist()
	print(f"\n  ── Cluster {cluster_id} ──")
	for s in samples:
		print(f"    • {s[:120]}")



  ── Cluster 0 ──
    • A Beginner's Guide to Prompt Engineering with GitHub Copilot When I started using GitHub Copilot and other generative AI
    • AI in the Wild: A ChatGPT Simulated Town 🤖🔥 Have you heard about a village where all its citizens are controlled by arti
    • Unveiling the Future: A Deep Dive into OpenAI's Groundbreaking o1 Reasoning Model Unveiling OpenAI’s New Reasoning Model
    • Beyond LLMs: My Introductory Experience with AI Agents If you ask ten developers to define an 'AI agent,' you'll get fif
    • Connecting AI Agents to Your Systems with MCP title: Connecting AI Agents to Your Systems with MCP published: true descr

  ── Cluster 1 ──
    • Save Money and Frustration on Amazon using Azure Functions Amazon is a service known for convenience and efficiency. How
    • Building a Scalable E-Commerce Data Model title: Building a Scalable E Commerce Data Model published: true description: 
    • Why SEO Is Important: Case Study One day, every micro, small, or me